# Task 2 — Hands-On Data Lab Implementation
**Candidate:** Ganesh  
**Program:** YuvaIntern Data Analyst Internship

## Project
Practical retail-sales data analysis using Python, Pandas, NumPy, and Matplotlib.

**Data note:** This is a synthetic dataset generated for learning and demonstration. It does not represent a real business.


## 1. Objectives
1. Set up a Python data-analysis environment.
2. Load, inspect, clean, and transform a CSV dataset.
3. Use Pandas and NumPy for descriptive analysis.
4. Create clear visualizations and communicate evidence-based findings.

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)


## 2. Load and inspect data
Keep the raw CSV unchanged. Inspect shape, column types, preview rows, and summary statistics before cleaning.

In [ ]:
raw = pd.read_csv("retail_sales_raw.csv")
print("Raw shape:", raw.shape)
display(raw.head())
print("\nColumn types:")
print(raw.dtypes)
print("\nDescriptive statistics:")
display(raw.describe(include="all").transpose())
print("\nMissing values by column:")
print(raw.isna().sum())
print("\nExact duplicate rows:", int(raw.duplicated().sum()))


## 3. Data quality checks and cleaning decisions
The raw file intentionally contains one duplicated row and missing values. This exercise drops exact duplicates, fills missing numeric values with the median, fills the missing region with the mode, parses dates, and recalculates revenue. In real projects, investigate the reason for missingness and document why each imputation is appropriate.

In [ ]:
sales = raw.drop_duplicates().copy()

sales["Quantity"] = pd.to_numeric(sales["Quantity"], errors="coerce")
sales["Unit_Price"] = pd.to_numeric(sales["Unit_Price"], errors="coerce")
sales["Discount_Rate"] = pd.to_numeric(sales["Discount_Rate"], errors="coerce").fillna(0)
sales["Quantity"] = sales["Quantity"].fillna(sales["Quantity"].median())
sales["Unit_Price"] = sales["Unit_Price"].fillna(sales["Unit_Price"].median())
sales["Region"] = sales["Region"].fillna(sales["Region"].mode()[0])
sales["Date"] = pd.to_datetime(sales["Date"], errors="coerce")

sales["Revenue"] = (sales["Quantity"] * sales["Unit_Price"] * (1 - sales["Discount_Rate"])).round(2)

assert sales["Order_ID"].is_unique
assert sales["Quantity"].gt(0).all()
assert sales["Unit_Price"].ge(0).all()
assert sales["Discount_Rate"].between(0, 1).all()
assert sales["Revenue"].ge(0).all()

print("Cleaned shape:", sales.shape)
print("Missing cells after cleaning:", int(sales.isna().sum().sum()))
print("Duplicate rows after cleaning:", int(sales.duplicated().sum()))
display(sales.head())
sales.to_csv("retail_sales_cleaned.csv", index=False)


## 4. Feature creation and analysis
Create a month feature and compare revenue across categories, products, and regions. Revenue is recalculated from quantity, unit price, and discount so the metric follows one consistent definition.

In [ ]:
sales["Month"] = sales["Date"].dt.to_period("M").astype(str)

category_summary = sales.groupby("Category").agg(
    Orders=("Order_ID", "count"),
    Units=("Quantity", "sum"),
    Revenue=("Revenue", "sum")
).sort_values("Revenue", ascending=False)

product_summary = sales.groupby("Product")["Revenue"].sum().sort_values(ascending=False)
region_summary = sales.groupby("Region")["Revenue"].sum().sort_values(ascending=False)

print("Total revenue:", round(sales["Revenue"].sum(), 2))
print("Average order revenue:", round(sales["Revenue"].mean(), 2))
print("\nRevenue by category:")
display(category_summary.round(2))
print("\nRevenue by product:")
display(product_summary.round(2))
print("\nRevenue by region:")
display(region_summary.round(2))


## 5. NumPy and descriptive analysis

In [ ]:
revenue_array = sales["Revenue"].to_numpy()
print("Revenue mean:", round(np.mean(revenue_array), 2))
print("Revenue median:", round(np.median(revenue_array), 2))
print("Revenue standard deviation (population):", round(np.std(revenue_array), 2))
print("Revenue minimum:", round(np.min(revenue_array), 2))
print("Revenue maximum:", round(np.max(revenue_array), 2))
print("Quantity-revenue correlation:", round(sales["Quantity"].corr(sales["Revenue"]), 3))


## 6. Visualization
Charts help compare categories and regions and explore the relationship between units sold and revenue. Axes are labelled and charts are generated from the cleaned data.

In [ ]:
category_summary["Revenue"].plot(kind="bar", figsize=(7, 4.5), title="Revenue by Product Category")
plt.xlabel("Category"); plt.ylabel("Revenue"); plt.tight_layout(); plt.show()

region_summary.plot(kind="bar", figsize=(7, 4.5), title="Revenue by Region")
plt.xlabel("Region"); plt.ylabel("Revenue"); plt.tight_layout(); plt.show()

plt.figure(figsize=(7, 4.5))
plt.scatter(sales["Quantity"], sales["Revenue"])
plt.title("Quantity vs Revenue")
plt.xlabel("Quantity"); plt.ylabel("Revenue")
plt.tight_layout(); plt.show()


## 7. Findings and interpretation
- Category and region rankings are calculated from the cleaned data, not assumed in advance.
- Order revenue depends on quantity, price, and discount; quantity alone will not fully explain revenue.
- Correlation indicates association only and does not prove that one variable causes another.
- The dataset is synthetic and small, so findings are a demonstration of method rather than business advice.

In [ ]:
print("Highest-revenue category:", category_summary["Revenue"].idxmax())
print("Highest-revenue product:", product_summary.idxmax())
print("Highest-revenue region:", region_summary.idxmax())
print("Correlation between quantity and revenue:", round(sales["Quantity"].corr(sales["Revenue"]), 3))


## 8. Best practices and limitations
1. Preserve the original raw data and create a separate cleaned dataset.
2. Check missing values, duplicate records, types, ranges, and calculation consistency.
3. Document imputation and feature-engineering choices.
4. Use readable column names and reproducible notebook cells.
5. Label charts clearly and do not overstate patterns.
6. Use authorized data and protect sensitive information.
7. For real business use, validate data definitions, investigate anomalies, and confirm results with stakeholders.

## 9. Conclusion
This lab demonstrates setting up and using a Python data-analysis environment, loading and cleaning tabular data, performing descriptive analysis, grouping data with Pandas, using NumPy, visualizing results with Matplotlib, and documenting findings and limitations. The dataset is synthetic; replace it with an approved real dataset if the task rubric requires real-world data.